In [1]:
import os
import pandas as pd
import numpy as np
from app_config import REPORTS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
from models.DeepConvLSTM import DeepConvLSTM, HARDataset, collate_fn, create_weighted_sampler
import optuna
import optuna.visualization as vis
import train_toys
#definisco il path da cui leggere i .csv

path='C:\codes\HumanActivityRecognition\data\pdd_data'
print(path)


df=pd.read_csv(os.path.join(path,'3002_BA.csv'))
print(df.shape)
df = df[df['action_id'] != 0]
print(df.shape)
print(df.columns)
#stampa il contenuto della colonna 'action'
print(df['action'].value_counts())
print(df['toy_id'].value_counts())


df=pd.read_csv(os.path.join(path,'3005_BA.csv'))
print(df.shape)
df = df[df['action_id'] != 0]
print(df.shape)
print(df.columns)
#stampa il contenuto della colonna 'action'
print(df['action'].value_counts())
print(df['toy_id'].value_counts())


#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path

df_list_ball = [] #lista vuota per appendere i dataframe con attività non nulla
for file in os.listdir(path):
    if file.endswith('.csv'):
        #leggo solo i file che dopo l'undescore ha BA
        if file.split('_')[1]=='BA.csv':
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_ball.append(df_temp)

df_ball = pd.concat(df_list_ball)
print("Dimensioni del df_ball con tutte le attività non nulle")
print(df_ball.shape)
print(df_ball.columns)
print(df_ball['action'].value_counts())

#salvo il dataframe
df_ball.to_csv(os.path.join(path,'df_BA_non_null.csv'),index=False)


#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_ball['action_id'].unique():
    df_action = df_ball[df_ball["action_id"] == action_id] #filtro il dataframe in base all'attività
    print(f"Dimensioni del dataframe df_ball_action_{action_id}")
    print(df_action.shape) #stampo le dimensioni del dataframe
    print(df_action.columns) #stampo le colonne del dataframe
    print(df_action['action'].value_counts()) #stampo il conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_ball_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    print(f"Salvato il dataframe df_ball_action_{action_id}.csv")


#applico sliding window con la funzion process_csv
#definisco i parametri
nb_sensor_channels = 13
sliding_window_length = 100
sliding_window_step = 20

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al toy palla
#e poi concateno tutto in un unica x_train, y_train

X= []
Y= []

for file in os.listdir(path):
    if file.endswith('.csv') and file.split('_')[1] == 'ball':
        file_path = os.path.join(path, file)
        print(file_path)
        X_windows, Y_windows = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)
        X.append(X_windows)
        Y.append(Y_windows)

# Concatenate all the windows into a single array
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)

#NUMERO TOTALE DI FINESTRE PER LA PALLA
print("Numero totale di finestre per il giocattolo ball:")
print(X.shape)
print(Y.shape)

#verifca
print(X)
print(Y)


# estraggo gli id dei bambini per vedere quanti ne ho
kid_ids = X[:, :, -2]  # Assuming kid_id is the third last column

# Get unique kid_ids
unique_kid_ids = np.unique(kid_ids)
print("Unique kid_ids in X:")
print(unique_kid_ids)



#Splitto il dataset in base al numero di azioni eseguite per avere congruenza temporale tra train e test e per 
#cercare di bilanciare le finestre in train e test

#filtro per ogni
#filtro per contare il numero di finestre per ogni azione
unique_actions, counts = np.unique(Y, return_counts=True)
print(unique_actions)
action_counts = dict(zip(unique_actions, counts))

print("Numero di finestre per ogni azione:")
for action, count in action_counts.items():
    print(f"Azione {action}: {count} finestre")


#split ratio  (70% nel train e 30% nel test)
split_ratio = 0.7

# Split the data
X_train = []
Y_train = []
X_test = []
Y_test = []

for action in unique_actions:

    #trovo gli indici delle finestre corrispondenti a ciascuna azione
    action_indices = np.where(Y == action)[0]
    print(action_indices)

    # Calcolo il numero di finestre da usare per il train e per il test
    num_windows = len(action_indices)
    num_train = int(num_windows * split_ratio)
    num_test = num_windows - num_train
    
    # Divido gli indici delle finestre in train e test
    train_indices = action_indices[:num_train]
    test_indices = action_indices[-num_test:]
    
    # Aggiungo le finestre al train e al test set
    X_train.append(X[train_indices])
    Y_train.append(Y[train_indices])
    X_test.append(X[test_indices])
    Y_test.append(Y[test_indices])

# Concateno tutti i dati in un unico array
X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)
X_test = np.concatenate(X_test, axis=0)
Y_test = np.concatenate(Y_test, axis=0)

print("Training set shape:", X_train.shape, Y_train.shape)
print (Y_train)
print("Test set shape:", X_test.shape, Y_test.shape)
print(Y_test)

#stampo il tipo di valore che contiene x_train e y_train(se int float ecc)
print("Tipo di X_train:", X_train.dtype)
print("Tipo di Y_train:", Y_train.dtype)

#stampo il tipo di x_train y train x test e y test
print("Tipo di X_train:", type(X_train))
print("Tipo di Y_train:", type(Y_train))
print("Tipo di X_test:", type(X_test))
print("Tipo di Y_test:", type(Y_test))

#rimuovo colonne in eccesso in x_train e x_test (rimangono solo le prime 9 colonne)
X_train = X_train[:, :, :9]
X_test = X_test[:, :, :9]
print("Dimensioni di X_train e X_test dopo aver rimosso le colonne in eccesso:")
print(X_train.shape, X_test.shape)

Y_train = Y_train.flatten()
Y_test = Y_test.flatten()

print("Etichette train:", Y_train)
print("Etichette test:", Y_test)

#tipo
print("Tipo di Y_train:", type(Y_train))
print("Tipo di Y_test:", type(Y_test))

# Trova tutte le etichette uniche presenti nei dati
unique_labels = np.unique(Y_train)

# Crea un dizionario che mappa ogni etichetta originale a un valore consecutivo
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}

# Stampa il dizionario per vedere il mapping
print("Mapping delle etichette:", label_mapping)

# Applica il mapping ai dataset di train e test
Y_train_mapped = np.array([label_mapping[y] for y in Y_train])
Y_test_mapped = np.array([label_mapping[y] for y in Y_test])

# Controllo finale
print("Nuove etichette train:", np.unique(Y_train_mapped))
print("Nuove etichette test:", np.unique(Y_test_mapped))


# Creo i dataset per il training e il test
train_dataset = HARDataset(X_train, Y_train_mapped)
test_dataset = HARDataset(X_test, Y_test_mapped)

 #stampo il dataset di training e di test a livello di dimensioni
print("Dataset di training:", len(train_dataset))
print("Dataset di test:", len(test_dataset))

print("Tipo di train_dataset:", type(train_dataset))
print("Tipo di test_dataset:", type(test_dataset))



# Creazione sampler pesato per il dataset di training
train_sampler = create_weighted_sampler(Y_train_mapped)







def objective(trial):
    # Definisci gli iperparametri da ottimizzare
    lr = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [4, 8, 12])

    # Crea i DataLoader con il batch_size suggerito
    #runno di nuovo il train con 5 secondi di finestra 
    train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True, drop_last=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

    # Crea il modello con gli iperparametri suggeriti# Carica il modello preaddestratocd
    model = DeepConvLSTM()

        # Rimuovi la testa originale, in modo da non caricare i pesi associati
    model.load_state_dict(torch.load('best_model_dl.pth'), strict=False)

    # Ora sostituisci la testa del modello con la nuova dimensione di classi (4)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, 4)  # 4 classi
    model.set_n_classes(4)

    # Congela tutti i parametri tranne quelli della testa (fully connected)
    for param in model.parameters():
        param.requires_grad = False  # Congela tutti i pesi

    # Sblocca i parametri della testa (fully connected)
    for param in model.fc.parameters():
        param.requires_grad = True  # Solo i pesi della testa saranno addestrabili

    # Esegui l'allenamento
    best_f1_score = train_toys.train(model, train_loader, test_loader, epochs=100, batch_size=batch_size, lr=lr)

    return best_f1_score

# Creazione studio Optuna ottimizza, nel senso di minimizzare la loss in 100 prove
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("Best hyperparameters: ", study.best_params)
print("Highest F1-score: ", study.best_value)


#salvo i best hyperparameters
best_hyperparameters = study.best_params
best_hyperparameters['best_f1_score'] = study.best_value
best_hyperparameters_df = pd.DataFrame([best_hyperparameters])
best_hyperparameters_df.to_csv(os.path.join(REPORTS_DIR, 'best_hyperparameters_ball.csv'), index=False)




#visualizzare la storia dell'ottimizzazione effettuata da Optuna. Ci permette di vedere come la loss
# è cambiata nel corso delle diverse prove (trials) durante l'ottimizzazione.
vis.plot_optimization_history(study)

2025-02-28 11:57:58.439 | INFO     | app_config:<module>:11 - PROJ_ROOT path is: C:\codes\HumanActivityRecognition


No GPU available, training on CPU; consider making n_epochs very small.


c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No GPU available, training on CPU; consider making n_epochs very small.
C:\codes\HumanActivityRecognition\data\pdd_data


C:\Users\carol\AppData\Local\Temp\ipykernel_1188\3139320034.py:19: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(os.path.join(path,'3002_BA.csv'))


...

Epoch: 30/100... Train Loss: 0.5872... Val Loss: 0.3501... Val Acc: 0.8500... F1-Score: 0.7952
Early stopping triggered
Epoch: 1/100... Train Loss: 1.3097... Val Loss: 1.0751... Val Acc: 0.5278... F1-Score: 0.4854
Epoch: 2/100... Train Loss: 0.9867... Val Loss: 0.9726... Val Acc: 0.6111... F1-Score: 0.5676
Epoch: 3/100... Train Loss: 0.8116... Val Loss: 0.8252... Val Acc: 0.6111... F1-Score: 0.4901
Epoch: 4/100... Train Loss: 0.7534... Val Loss: 0.6010... Val Acc: 0.7500... F1-Score: 0.6487
Epoch: 5/100... Train Loss: 0.6824... Val Loss: 0.5466... Val Acc: 0.8333... F1-Score: 0.7633
Epoch: 6/100... Train Loss: 0.7127... Val Loss: 0.5539... Val Acc: 0.7778... F1-Score: 0.6975
Epoch: 7/100... Train Loss: 0.6125... Val Loss: 0.5367... Val Acc: 0.8056... F1-Score: 0.7381
Epoch: 8/100... Train Loss: 0.7364... Val Loss: 0.5119... Val Acc: 0.8333... F1-Score: 0.7698
Epoch: 9/100... Train Loss: 0.6760... Val Loss: 0.5507... Val Acc: 0.7500... F1-Score: 0.6592
Epoch: 10/100... Train Loss: 0.751

[I 2025-02-28 12:02:36,396] Trial 37 finished with value: 0.8809523809523809 and parameters: {'lr': 0.028405953796484376, 'batch_size': 12}. Best is trial 33 with value: 0.9285714285714285.


Epoch: 35/100... Train Loss: 0.6391... Val Loss: 0.3377... Val Acc: 0.8611... F1-Score: 0.8002
Epoch: 36/100... Train Loss: 0.6216... Val Loss: 0.3913... Val Acc: 0.8889... F1-Score: 0.8408
Epoch: 37/100... Train Loss: 0.6347... Val Loss: 0.5183... Val Acc: 0.7500... F1-Score: 0.6604
Early stopping triggered
Epoch: 1/100... Train Loss: 1.1973... Val Loss: 1.0619... Val Acc: 0.6389... F1-Score: 0.6462
Epoch: 2/100... Train Loss: 1.0172... Val Loss: 0.8577... Val Acc: 0.5833... F1-Score: 0.4514
Epoch: 3/100... Train Loss: 0.7773... Val Loss: 0.7181... Val Acc: 0.6389... F1-Score: 0.5258


[W 2025-02-28 12:02:36,803] Trial 38 failed with parameters: {'lr': 0.050570636447539064, 'batch_size': 12} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\optuna\study\_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\carol\AppData\Local\Temp\ipykernel_1188\3139320034.py", line 272, in objective
    best_f1_score = train_toys.train(model, train_loader, test_loader, epochs=100, batch_size=batch_size, lr=lr)
  File "c:\codes\HumanActivityRecognition\HumanActivityRecognition\train_toys.py", line 79, in train
    output, val_h = net(inputs, val_h, batch_size)
  File "c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\torch\nn\modules\module.py", line 1739, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
  File "c:\Users\carol\Desktop\virtual_envs\har_venv\lib\site-packages\torch\nn\modules\module.py", line 1750

KeyboardInterrupt: 

In [ ]:
for name, param in model.named_parameters():
    print(f"{name} requires_grad= {param.requires_grad}")


In [ ]:
import os
import pandas as pd
import numpy as np
from app_config import PROJ_ROOT, DATA_DIR, REPORTS_DIR
import sliding_window_on_data
from torch.utils.data import DataLoader
import torch
from models.DeepConvLST_toys import DeepConvLSTM, HARDataset, collate_fn, create_weighted_sampler
import optuna
import optuna.visualization as vis
import train_toys

from utils.transformations import *
from utils.transformations_utils import *
    

path='C:\codes\HumanActivityRecognition\data\pdd_data'
print(path)


df=pd.read_csv(os.path.join(path,'3002_BA.csv'))
print(df.shape)
df = df[df['action_id'] != 0]
print(df.shape)
print(df.columns)
#stampa il contenuto della colonna 'action'
print(df['action'].value_counts())
print(df['toy_id'].value_counts())


df=pd.read_csv(os.path.join(path,'3005_BA.csv'))
print(df.shape)
df = df[df['action_id'] != 0]
print(df.shape)
print(df.columns)
#stampa il contenuto della colonna 'action'
print(df['action'].value_counts())
print(df['toy_id'].value_counts())


#Visto che l'informazione relativa all'id del bambino lo abbiamo, così come abbiamo anche l'informazione relativa
#al giocattolo, vado a filtrare i .csv in modo tale da avere solo le righe che hanno l'attività non nulla e poi
#appendo tutte le righe delle righe non nulle in un unico dataframe
#per tutti i file che terminano in .csv nella cartella path

df_list_ball = [] #lista vuota per appendere i dataframe con attività non nulla
for file in os.listdir(path):
    if file.endswith('.csv'):
        #leggo solo i file che dopo l'undescore ha BA
        if file.split('_')[1]=='BA.csv':
            df_temp=pd.read_csv(os.path.join(path,file))
            df_temp = df_temp[df_temp['action_id'] != 0]
            df_list_ball.append(df_temp)

df_ball = pd.concat(df_list_ball)
print("Dimensioni del df_ball con tutte le attività non nulle")
print(df_ball.shape)
print(df_ball.columns)
print(df_ball['action'].value_counts())

#salvo il dataframe
df_ball.to_csv(os.path.join(path,'df_ball_non_null.csv'),index=False)


#ora divido il dataframe in base all'attività (action_id) e salvo i dataframe in un file .csv
#per ogni attività

for action_id in df_ball['action_id'].unique():
    df_action = df_ball[df_ball["action_id"] == action_id] #filtro il dataframe in base all'attività
    print(f"Dimensioni del dataframe df_ball_action_{action_id}")
    print(df_action.shape) #stampo le dimensioni del dataframe
    print(df_action.columns) #stampo le colonne del dataframe
    print(df_action['action'].value_counts()) #stampo il conteggio delle attività
    #salvo il dataframe
    df_action.to_csv(os.path.join(path,f'df_ball_action_{action_id}.csv'),index=False) #index=False per non salvare l'indice
    print(f"Salvato il dataframe df_ball_action_{action_id}.csv")


#applico sliding window con la funzion process_csv
#definisco i parametri
nb_sensor_channels = 13
sliding_window_length = 100
sliding_window_step = 20

#ora applico la funzione sliding window (che mi da come output x_window e y_window) a tutti i .csv relativi al toy palla
#e poi concateno tutto in un unica x_train, y_train

X= []
Y= []

for file in os.listdir(path):
    if file.endswith('.csv') and file.split('_')[1] == 'ball':
        file_path = os.path.join(path, file)
        print(file_path)
        X_windows, Y_windows = sliding_window_on_data.process_csv(file_path, nb_sensor_channels, sliding_window_length, sliding_window_step)
        X.append(X_windows)
        Y.append(Y_windows)

# Concatenate all the windows into a single array
X = np.concatenate(X, axis=0)
Y = np.concatenate(Y, axis=0)

#NUMERO TOTALE DI FINESTRE PER LA PALLA
print("Numero totale di finestre per il giocattolo ball:")
print(X.shape)
print(Y.shape)

#verifca
print(X)
print(Y)


# estraggo gli id dei bambini per vedere quanti ne ho
kid_ids = X[:, :, -2]  # Assuming kid_id is the third last column

# Get unique kid_ids
unique_kid_ids = np.unique(kid_ids)
print("Unique kid_ids in X:")
print(unique_kid_ids)



#Splitto il dataset in base al numero di azioni eseguite per avere congruenza temporale tra train e test e per 
#cercare di bilanciare le finestre in train e test

#filtro per ogni
#filtro per contare il numero di finestre per ogni azione
unique_actions, counts = np.unique(Y, return_counts=True)
print(unique_actions)
action_counts = dict(zip(unique_actions, counts))

print("Numero di finestre per ogni azione:")
for action, count in action_counts.items():
    print(f"Azione {action}: {count} finestre")


#split ratio  (70% nel train e 30% nel test)
split_ratio = 0.7

# Split the data
X_train = []
Y_train = []
X_test = []
Y_test = []

for action in unique_actions:

    #trovo gli indici delle finestre corrispondenti a ciascuna azione
    action_indices = np.where(Y == action)[0]
    print(action_indices)

    # Calcolo il numero di finestre da usare per il train e per il test
    num_windows = len(action_indices)
    num_train = int(num_windows * split_ratio)
    num_test = num_windows - num_train
    
    # Divido gli indici delle finestre in train e test
    train_indices = action_indices[:num_train]
    test_indices = action_indices[-num_test:]
    
    # Aggiungo le finestre al train e al test set
    X_train.append(X[train_indices])
    Y_train.append(Y[train_indices])
    X_test.append(X[test_indices])
    Y_test.append(Y[test_indices])

# Concateno tutti i dati in un unico array
X_train = np.concatenate(X_train, axis=0)
Y_train = np.concatenate(Y_train, axis=0)
X_test = np.concatenate(X_test, axis=0)
Y_test = np.concatenate(Y_test, axis=0)

print("Training set shape:", X_train.shape, Y_train.shape)
print (Y_train)
print("Test set shape:", X_test.shape, Y_test.shape)
print(Y_test)

#stampo il tipo di valore che contiene x_train e y_train(se int float ecc)
print("Tipo di X_train:", X_train.dtype)
print("Tipo di Y_train:", Y_train.dtype)

#stampo il tipo di x_train y train x test e y test
print("Tipo di X_train:", type(X_train))
print("Tipo di Y_train:", type(Y_train))
print("Tipo di X_test:", type(X_test))
print("Tipo di Y_test:", type(Y_test))

Y_train = Y_train.flatten()
Y_test = Y_test.flatten()

print("Etichette train:", Y_train)
print("Etichette test:", Y_test)

#tipo
print("Tipo di Y_train:", type(Y_train))
print("Tipo di Y_test:", type(Y_test))

# Trova tutte le etichette uniche presenti nei dati
unique_labels = np.unique(Y_train)

# Crea un dizionario che mappa ogni etichetta originale a un valore consecutivo
label_mapping = {label: idx for idx, label in enumerate(unique_labels)}

# Stampa il dizionario per vedere il mapping
print("Mapping delle etichette:", label_mapping)

# Applica il mapping ai dataset di train e test
Y_train_mapped = np.array([label_mapping[y] for y in Y_train])
Y_test_mapped = np.array([label_mapping[y] for y in Y_test])

# Controllo finale
print("Nuove etichette train:", np.unique(Y_train_mapped))
print("Nuove etichette test:", np.unique(Y_test_mapped))


print(f"Dimensioni di X_Train: {X_train.shape}")
print(f"Dimensioni di Y_Train: {Y_train_mapped.shape}")



train_dataset = HARDataset(X_train, Y_train_mapped)
test_dataset = HARDataset(X_test, Y_test_mapped)
# Creo i dataset per il training e il test


# Creazione sampler pesato per il dataset di training
train_sampler = create_weighted_sampler(Y_train_mapped)









    # Crea i DataLoader con il batch_size suggerito
    #runno di nuovo il train con 5 secondi di finestra 
train_loader = DataLoader(train_dataset, batch_size=8,shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, drop_last=True)

    # Crea il modello con gli iperparametri suggeriti# Carica il modello preaddestratocd
model = DeepConvLSTM()

        # Rimuovi la testa originale, in modo da non caricare i pesi associati
model.load_state_dict(torch.load('best_model_dl.pth'), strict=False)

    # Ora sostituisci la testa del modello con la nuova dimensione di classi (4)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 4)  # 4 classi
model.set_n_classes(4)

    # Congela tutti i parametri tranne quelli della testa (fully connected)
for param in model.parameters():
        param.requires_grad = False  # Congela tutti i pesi

    # Sblocca i parametri della testa (fully connected)
for param in model.fc.parameters():
    param.requires_grad = True  # Solo i pesi della testa saranno addestrabili



file_path = os.path.join(REPORTS_DIR, 'best_hyperparameters_ball.csv')
loaded_params_df = pd.read_csv(file_path)
loaded_params = loaded_params_df.iloc[0].to_dict()  # Convertola prima riga in un dizionario

# Estrai i valori
lr = loaded_params["lr"]
bs = int(loaded_params["batch_size"])  # Assicurati che sia un intero

print(loaded_params)
best_f1_score = train_toys.train(model, train_loader, test_loader, epochs=100, batch_size= bs, lr=lr,cm_filename= "CM INFERENCE BALL")
print(f"Best F1 score: {best_f1_score}")

C:\codes\HumanActivityRecognition\data\pdd_data


C:\Users\carol\AppData\Local\Temp\ipykernel_1188\3080109252.py:21: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv(os.path.join(path,'3002_BA.csv'))


(161440, 25)
(1513, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
action
impila          665
lancia          420
sposta posto    248
afferra         180
Name: count, dtype: int64
toy_id
BA    1513
Name: count, dtype: int64
(89208, 25)
(0, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
Series([], Name: count, dtype: int64)
Series(

C:\Users\carol\AppData\Local\Temp\ipykernel_1188\3080109252.py:51: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp=pd.read_csv(os.path.join(path,file))


Dimensioni del df_ball con tutte le attività non nulle
(1513, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
action
impila          665
lancia          420
sposta posto    248
afferra         180
Name: count, dtype: int64
Dimensioni del dataframe df_ball_action_21
(665, 25)
Index(['Timestamp', 'Accel_LN_X', 'Accel_LN_Y', 'Accel_LN_Z', 'Accel_WR_X',
       'Accel_WR_Y', 'Accel_WR_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Mag_X',
       'Mag_Y', 'Mag_Z', 'Date_time', 'action', 'action_id', 'G', 'kid_id',
       'toy_id', 'communication', 'social_interaction',
       'restricted_repetitive_behaviour', 'ados_total_score', 'I', 'E'],
      dtype='object')
action
impila 

RuntimeError: Error(s) in loading state_dict for DeepConvLSTM:
	size mismatch for conv1.weight: copying a param with shape torch.Size([64, 9, 5]) from checkpoint, the shape in current model is torch.Size([64, 13, 5]).
	size mismatch for fc.weight: copying a param with shape torch.Size([29, 128]) from checkpoint, the shape in current model is torch.Size([4, 128]).
	size mismatch for fc.bias: copying a param with shape torch.Size([29]) from checkpoint, the shape in current model is torch.Size([4]).